In [ ]:
import pandas as pd
import requests
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Gemini Recomendations
##### Add FCF Yield	In a high-interest-rate environment, the ability to generate Free Cash Flow (FCF) is paramount. FCF Yield (MarketCap/FCF) is a better valuation metric than P/E for high-growth tech firms.

##### Compare Alpha to Sector ETF	Don't just check performance against the Nifty/S&P. Check if the stock's Alpha is positive relative to its Sector ETF. This isolates company-specific success from sector tailwinds.

##### Integrate Analyst Revisions	Overlay data on whether analyst Target Prices and Earnings Estimates have been recently revised upwards. This is a strong leading indicator ahead of the earnings season.

In [ ]:
# === STEP 1: DATA FETCHING ===

def fetch_stock_data(symbols, api_func, **kwargs):
    data = []
    for symbol in symbols:
        result = api_func(symbol, **kwargs)
        if result:
            data.append(result)
    return pd.DataFrame(data)

# Example API function (replace with your data provider/API)
def example_api(symbol):
    # Placeholder: Replace with actual API logic. Could be Yahoo Finance, etc.
    return {
        'symbol': symbol,
        'marketCap': np.random.uniform(1e9, 1e13),
        'peRatio': np.random.uniform(10, 30),
        'returnOnEquityTTM': np.random.uniform(0.15, 1.2),
        'debtToEquity': np.random.uniform(0, 1),
        'growthRevenue': np.random.uniform(-0.3, 2.5),
        'growthNetIncome': np.random.uniform(-0.3, 2),
        'industry': np.random.choice(['IT', 'Pharma', 'Banking', 'Electronics', 'Semiconductors']),
        'companyName': f'Company {symbol}',
    }

SYMBOL_LIST = ['TCS.NS', 'HCLTECH.NS', 'DIXON.NS', 'KPITTECH.NS', 'DRREDDY.NS', 'VINTRON.BO', 'INA.BO', 'CNCRD.BO', 'PGHL.NS']
df = fetch_stock_data(SYMBOL_LIST, example_api)

In [ ]:
# === STEP 2: SCREENING CRITERIA ===

criteria = (
    (df['marketCap'] > 1e10) &
    (df['peRatio'].between(10, 30)) &
    (df['returnOnEquityTTM'] > 0.15) &
    (df['debtToEquity'] < 1) &
    (df['growthRevenue'] > 0.1)
)
screened = df[criteria].copy()

In [ ]:
# === STEP 3: SECTOR ANALYSIS ===

sector_groups = screened.groupby('industry').agg({
    'returnOnEquityTTM': 'mean',
    'growthRevenue': 'mean',
    'peRatio': 'mean'
}).reset_index()

plt.figure(figsize=(10,6))
melted = sector_groups.melt(id_vars='industry', var_name='metric', value_name='average')
sns.barplot(x='industry', y='average', hue='metric', data=melted)
plt.title('Average Sector Metrics')
plt.show()

In [ ]:
# === STEP 4: COMPOSITE SCORING ===

screened['composite_score'] = (
    screened['returnOnEquityTTM'] * 0.3 +
    screened['growthRevenue'] * 0.25 +
    screened['growthNetIncome'] * 0.25 +
    (1 / screened['peRatio']) * 0.2
)

top_picks = screened.nlargest(10, 'composite_score')
print("Top 10 Investment Picks:\n", top_picks[['symbol','companyName','industry','composite_score']])

In [ ]:
# === STEP 5: FINAL RECOMMENDATION ===

def get_recommendation(top_picks, cap_type='balanced'):
    if cap_type == 'aggressive':
        return top_picks.head(4)
    elif cap_type == 'conservative':
        return top_picks[top_picks['marketCap'] > 1e12]
    else:
        return top_picks.head(7)

final_recommendation = get_recommendation(top_picks)
print("\nFinal Stock Recommendation:\n", final_recommendation[['symbol','companyName','industry']])

In [ ]:
# === STEP 6: RISK FACTOR REPORTING (Sample) ===

risk_factors = [
    "Global volatility: FOMC minutes, geopolitical tensions",
    "Currency risk: Rupee vs Dollar",
    "Valuation: High PE segments",
    "Earnings slowdown: Watch Q2 season"
]
print("\nRisk Factors to Monitor:")
for item in risk_factors:
    print("- " + item)

# === STEP 7: AUTOMATED DOCUMENTATION ===
final_recommendation.to_csv("recommended_stocks.csv", index=False)
print("\nAnalysis CSV 'recommended_stocks.csv' saved.")